# 강의 04 · 실습 8 — 이미지 생성 파이프라인 · (6) 고난도 3 — 두 버전 비교와 판정표 기록

## 1. 문제상황

- 팀은 지시문 템플릿을 개정할 때마다 v1과 v2로 같은 주제의 이미지를 뽑아 나란히 비교합니다.
- 비교 결과와 근거는 판정표에 남겨야 다음 개정에 쓸 수 있는데, 지금은 디자이너가 그림을 본 뒤 표를 손으로 채웁니다.
- 판정 근거를 쓰는 데 시간이 걸려서 표가 비어 있는 채로 남는 날이 많습니다.
- 팀은 판정 근거의 초안을 모델이 이미지를 읽어 써 주고, 사람은 우승만 고르고, 표는 프로그램이 파일에 적기로 정했습니다.

## 2. 문제와 목표

- **문제**: 판정표를 사람이 손으로 채우다 보니 표가 비어 남습니다. 근거 초안은 이미지를 읽는 모델이 쓸 수 있고, 표에 적는 일은 프로그램이 할 수 있습니다.
- **목표**: 주제를 입력하면 프로그램이 템플릿 두 벌로 지시문 두 개를 쓰고 이미지를 2장 뽑고, 이미지를 읽는 모델로 각 이미지의 설명을 만들어 사람에게 후보·설명과 함께 보내고, 사람이 우승(v1 또는 v2)을 고르면 판정표 파일에 행을 적는 처리 흐름을 만듭니다. 사람이 「재설계」라고 답하면 지시문 설계부터 다시 돕니다.
    - 템플릿 두 벌: v1과 조명 묘사 구절을 더한 v2이며, 문면은 「6. 코드 — 스텝바이스텝」 단계 0에 주어져 있습니다. 주제는 「비 내리는 밤 서울 골목의 LP 바 창가」입니다.
    - 이미지를 읽는 모델: 이미지를 넣어 두 문장 설명을 받는 함수 `vision(path)`이며 단계 0에 준비되어 있습니다.
    - 판정표: `out_images` 폴더 안의 텍스트 파일이며, 첫 줄은 주제이고, 행 하나는 버전·바꾼 변수(v1 「기준 버전」, v2 「조명 묘사 구절 추가」)·판정(우승 버전에만 「우승」)·근거(모델의 설명)입니다. 사람의 답(「v2」)은 코드에 대본으로 미리 정해 넣습니다.
- **목표 달성 여부의 판정 기준**
    - 우승을 답한 뒤 판정표 파일에 v1 행과 v2 행이 적힙니다.
    - 우승 행의 판정 칸에 「우승」이, 근거 칸에 모델이 쓴 설명이 들어 있습니다.
    - 그래프가 END에 도달하는 것을 실행 결과와 파일 내용에서 확인합니다.


## 3. 워크플로우 다이어그램

## 4. 단계별 요구사항

(학생이 번호 목록으로 씁니다.)

## 5. 코드 골격

이 실습의 코드 골격을 직접 세웁니다. 단계 · 하는 일 · 사용하는 코드 · 대응하는 요구사항 네 칸 표로 적습니다.

## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

라이브러리를 불러오고 텍스트 모델과 이미지 모델을 준비합니다. 이미지 모델을 부르는 함수 `paint`도 여기서 정의합니다.

- API 키와 자격증명은 `.env` 파일에서 읽습니다. `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 둡니다.
- `.env` 파일에는 다음 네 줄이 있어야 합니다. 값은 각자 발급받은 것을 넣습니다.

```
OPENAI_API_KEY=발급받은_키
GOOGLE_APPLICATION_CREDENTIALS=서비스_계정_키_파일의_경로
VERTEX_PROJECT=프로젝트_이름
VERTEX_LOCATION=리전_이름
```

- `paint` 호출 1회가 이미지 1장이고, 호출마다 비용이 듭니다. 생성한 이미지는 노트북 옆의 `out_images` 폴더에 저장됩니다.
- `show`는 후보 이미지 경로 목록을 화면에 표시하는 보조 함수입니다.
- 생성 호출 횟수는 전역 카운터 `paint_calls`로 셉니다.
- 이 실습에서는 이미지를 읽는 함수 `vision`도 여기서 정의합니다.

In [ ]:
import os
import time
from operator import add
from pathlib import Path

from dotenv import load_dotenv, find_dotenv
from typing import Annotated, TypedDict

from IPython.display import Image, display
from google import genai
from google.genai import types
from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt

load_dotenv(find_dotenv(usecwd=True))
for key in ("OPENAI_API_KEY", "GOOGLE_APPLICATION_CREDENTIALS", "VERTEX_PROJECT", "VERTEX_LOCATION"):
    if not os.environ.get(key):
        raise SystemExit(f"agentic-ai 폴더의 .env 파일에 {key} 줄을 넣습니다.")

llm = init_chat_model("openai/gpt-5.6-luna", model_provider="litellm")
client = genai.Client(vertexai=True, project=os.environ["VERTEX_PROJECT"], location=os.environ["VERTEX_LOCATION"])
IMG_MODEL = "gemini-3.1-flash-lite-image"
OUT_DIR = Path("out_images")
OUT_DIR.mkdir(exist_ok=True)
paint_calls = 0


def paint(prompt: str, tag: str) -> str:
    """지시문을 이미지 모델에 보내 이미지 파일을 저장하고 경로를 돌려준다. 호출 1회 = 이미지 1장 = 비용 발생."""
    global paint_calls
    for attempt in range(4):
        try:
            res = client.models.generate_content(
                model=IMG_MODEL, contents=prompt,
                config=types.GenerateContentConfig(response_modalities=["IMAGE", "TEXT"]))
            break
        except Exception as e:
            if "429" in str(e) and attempt < 3:
                print(f"    [paint] 분당 한도 초과 — {30 * (attempt + 1)}초 뒤 다시 부릅니다")
                time.sleep(30 * (attempt + 1))
                continue
            raise
    paint_calls += 1
    part = [p for p in res.candidates[0].content.parts if p.inline_data][0].inline_data
    ext = "png" if "png" in part.mime_type else "jpg"
    path = OUT_DIR / f"{tag}_{paint_calls:02d}.{ext}"
    path.write_bytes(part.data)
    print(f"    [paint] {path.as_posix()} ({len(part.data)} bytes)")
    return path.as_posix()


def show(paths: list) -> None:
    """후보 이미지 경로 목록을 화면에 차례로 표시한다."""
    for i, p in enumerate(paths, 1):
        print(f"    후보 {i}: {p}")
        display(Image(filename=p, width=320))


print("모델 준비를 마쳤습니다. 이미지 저장 폴더:", OUT_DIR)

def vision(path: str) -> str:
    """이미지 파일을 같은 모델에 넣어 두 문장 설명을 받는다. 호출 1회 = 비용 발생."""
    data = Path(path).read_bytes()
    mime = "image/png" if path.endswith(".png") else "image/jpeg"
    for attempt in range(4):
        try:
            res = client.models.generate_content(
                model=IMG_MODEL,
                contents=[types.Part.from_bytes(data=data, mime_type=mime), "이 이미지를 두 문장으로 설명하라."])
            break
        except Exception as e:
            if "429" in str(e) and attempt < 3:
                print(f"    [vision] 분당 한도 초과 — {30 * (attempt + 1)}초 뒤 다시 부릅니다")
                time.sleep(30 * (attempt + 1))
                continue
            raise
    return (res.text or "").strip()

# 주어진 자료 — 지시문 템플릿 두 벌 (주제 문장을 뒤에 이어 붙여 언어 모델에 넣는다)
SPEC_V1 = ("다음 주제로 이미지 생성 지시문을 한 문단으로 쓴다. "
           "장면·조명·화각을 포함한다. 주제: ")
SPEC_V2 = ("다음 주제로 이미지 생성 지시문을 한 문단으로 쓴다. "
           "장면·조명·화각을 포함한다. 조명은 광원의 방향과 색온도를 한 구절로 적는다. 주제: ")


In [ ]:
# 여기에 코드를 작성합니다. 단계마다 셀을 나누어 작성합니다.


## 7. 실행 결과 확인

1. 주제 하나로 이미지 2장이 만들어지고, 각 이미지에 대한 모델의 설명 두 개가 사람에게 후보와 함께 전달된 채 멈춥니다.
2. 우승(v1 또는 v2)을 답하면 그래프가 끝나고, 판정표 파일에 주제 줄과 행 2개가 적힙니다.
3. 우승한 버전의 행에만 「우승」이 있고, 두 행의 근거 칸에 모델의 설명이 들어 있습니다.
4. 이미지 생성 호출과 이미지 읽기 호출이 각각 2회입니다.